# Exploratory Data Analysis

This notebook explores the cleaned tables from `data/03_cleaning/` to characterize distributions, identify patterns, and develop features for the pre/post analysis in `05_analysis.ipynb`. The pre/post split is 2020–2024 (baseline) vs. 2025 (treatment).

## Data Dictionary

### events — 161,404 rows, one per warning event

| Field | Type | Nulls | Description | Analysis notes |
|---|---|---|---|---|
| `wfo` | str | 0 | NWS Weather Forecast Office call sign (e.g. `OUN`) | 122 offices; consider CONUS-only filter for main analysis |
| `year` | int | 0 | Calendar year 2020–2025 | Pre: 2020–2024; Post: 2025 |
| `phenomena` | str | 0 | VTEC code: `TO`, `SV`, or `FF` | Always split metrics by phenomena — performance baselines differ substantially |
| `eventid` | int | 0 | API event identifier; unique within WFO-year-phenomena | Use `product_id` for joins, not `eventid` alone |
| `product_id` | str | 0 | Derived join key: `{year}{wfo}{eventid}{phenomena}W1` | FK target for `stormreports.events`; 99.97% match rate |
| `issue` | datetime UTC | 0 | Warning issuance time | Use for time-of-day and seasonal stratification |
| `expire` | datetime UTC | 0 | Warning expiration time | — |
| `duration_min` | float | 0 | Warning duration in minutes (`expire − issue`) | Derived; check for 0-minute and extreme values |
| `status` | str | 0 | Terminal status: `EXP`, `CAN`, `COR`, `EXT`, `CON`, `NEW` | `CAN` = cancelled before expiry; relevant to FAR interpretation |
| `verify` | bool | 0 | `True` if ≥1 confirming LSR matched | Primary outcome for POD; 46.3% of events verified |
| `lead0` | float (min) | 53.7% | Lead time in minutes from warning issuance to first confirming LSR; 0 = warning issued in the same minute as LSR (POD₂ semantics) | Null iff `verify=False`; primary lead time metric; 5.4% of verified events have `lead0=0` (timestamp resolution artifact); capped at 99th pct per phenomena |
| `lead0_capped` | bool | 0 | `True` if `lead0` was capped | Exclude or flag capped rows in lead time distribution analysis |

### stormreports — 236,800 rows, one per Local Storm Report

| Field | Type | Nulls | Description | Analysis notes |
|---|---|---|---|---|
| `wfo` | str | 0 | NWS Weather Forecast Office call sign | — |
| `year` | int | 0 | Calendar year 2020–2025 | Pre: 2020–2024; Post: 2025 |
| `valid` | datetime UTC | 0 | Time the storm report was filed | — |
| `lsrtype` | str | 0 | LSR phenomena: `TO`, `SV`, or `FF` | Always split by lsrtype |
| `typetext` | str | 0 | Human-readable report type description | Useful for EDA; not needed for POD/FAR |
| `warned` | bool | 0 | `True` if a matching warning was in effect | Defines the warned/unwarned split for POD calculation |
| `leadtime` | float (min) | 23.2% | Minutes from warning issuance to report time | Null iff `warned=False`; capped at 99th pct per lsrtype; secondary to `events.lead0` |
| `leadtime_capped` | bool | 0 | `True` if `leadtime` was capped | Flag in any leadtime distribution analysis |
| `events` | str | 19.9% | Comma-separated VTEC product IDs of matched warnings | Null iff `warned=False`; links report to `events.product_id` |
| `tdq` | bool | 0 | "Too Difficult to Qualify" — NWS could not confirm hazardous event | Exclude from POD numerator; 1.9% of reports (concentrated in SV) |
| `source` | str | 0 | Normalized report source (e.g. `TRAINED SPOTTER`, `PUBLIC`) | Potential stratification variable for source-bias analysis |
| `city` | str | 0 | City of report location; Nominatim-imputed where originally null | — |
| `county` | str | 0 | County of report location; Nominatim-imputed where originally null | — |
| `state` | str | 0 | Two-letter state code | Contains junk values (`'  '`, `'X '`, `'XX'`) — clean before use |
| `lon0` | float | 0 | Longitude of report location (exact point geometry) | Retained for spatial EDA |
| `lat0` | float | 0 | Latitude of report location (exact point geometry) | Retained for spatial EDA |
| `remark` | str | 11.5% | Free-text observer narrative | Not used in quantitative analysis |
